# EDA: action candidates (analytics/actions)

Every table below is **fixture output, not a result**: the games are synthetic (analytics/fixtures.py when importable, else tests/_fx_actions.py). Re-run on the owner's Mac with `games = validate.load_data_root(DATA_ROOT)` for the real tables. Held-out games are dropped before anything is computed; no plot library is used (tables and markdown only).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / 'analytics').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from IPython.display import Markdown, display
from analytics.actions import eda, features as F, validate as V

games = V.load_fixture_games(seed=0, n_games=4, heldout_every=0, n_possessions=16)
games = [g for g in games if not (g['heldout'] or V.is_heldout_id(g['sportvu_game']))]
cands, feats, recall, skipped = F.build_season(games)
n_poss = sum(len(g['possessions']) for g in games)
print(len(games), 'games', n_poss, 'possessions', len(cands), 'candidates', skipped, 'skipped possessions')

## Candidate counts per possession (fixture output, not a result)

In [ ]:
display(Markdown(eda.markdown_table(eda.candidate_counts_per_possession(cands, n_poss))))

## Class balance and rule class against label class (fixture output, not a result)

In [ ]:
bal = eda.class_balance(cands)
display(Markdown(eda.markdown_table(bal['classes'])))
display(Markdown(eda.markdown_table(bal['rule_vs_label'])))
display(Markdown(eda.markdown_table(recall)))

## Feature distributions by label class (fixture output, not a result)

One row per feature and class: n, mean, std and quartiles. `def_recovery_ft` is the one declared post-window feature.

In [ ]:
dist = eda.feature_distributions_by_class(cands, feats)
display(Markdown(eda.markdown_table(dist)))

## Shuffled-label check (fixture output, not a result)

Out-of-fold ball_screen F1 of the hgb model on the real labels against labels permuted within each game. A model that scores well on permuted labels is reading something it should not; the permuted row should sit near the base rate.

In [ ]:
display(Markdown(eda.markdown_table(eda.shuffled_label_check(cands, feats, seed=0, n_folds=min(4, len(games))))))